# Semantic Search Comparison

# Import Required Libraries

In [1]:
import pandas as pd

from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

seed = 42

# Create FAQ Dataset

In [2]:
faq = pd.DataFrame({
    "doc_id": [1, 2, 3, 4, 5, 6],
    "title": [
        "Reset password",
        "Failed payment",
        "Refund request",
        "Account verification",
        "Delivery tracking",
        "Damaged product",
    ],
    "text": [
        "Learn how to reset your password if you cannot log in.",
        "What to do when money is deducted but the payment fails.",
        "Steps to request a refund for an eligible order.",
        "How to verify your account using an email code.",
        "Track the current delivery status of your order.",
        "Report a damaged item and request a replacement.",
    ],
})

faq["full_text"] = (
    faq["title"]
    + " "
    + faq["text"]
)

display(faq)

,doc_id,title,text,full_text
0,1,Reset password,Learn how to reset your password if you cannot...,Reset password Learn how to reset your passwor...
1,2,Failed payment,What to do when money is deducted but the paym...,Failed payment What to do when money is deduct...
2,3,Refund request,Steps to request a refund for an eligible order.,Refund request Steps to request a refund for a...
3,4,Account verification,How to verify your account using an email code.,Account verification How to verify your accoun...
4,5,Delivery tracking,Track the current delivery status of your order.,Delivery tracking Track the current delivery s...
5,6,Damaged product,Report a damaged item and request a replacement.,Damaged product Report a damaged item and requ...


# Build TF-IDF Index

In [3]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
)

documentMatrix = vectorizer.fit_transform(
    faq["full_text"]
)

print("Documents:", documentMatrix.shape[0])
print("Features:", documentMatrix.shape[1])

Documents: 6
Features: 63


# Search With TF-IDF

In [4]:
def searchTfidf(query, topK=3):
    queryVector = vectorizer.transform([query])

    scores = cosine_similarity(
        queryVector,
        documentMatrix,
    ).ravel()

    results = faq[
        ["doc_id", "title", "text"]
    ].copy()

    results["tfidf_score"] = scores

    return (
        results
        .sort_values(
            "tfidf_score",
            ascending=False,
        )
        .head(topK)
        .reset_index(drop=True)
    )


query = "money taken but order failed"
tfidfResults = searchTfidf(query)

display(tfidfResults)

,doc_id,title,text,tfidf_score
0,2,Failed payment,What to do when money is deducted but the paym...,0.339317
1,5,Delivery tracking,Track the current delivery status of your order.,0.107384
2,3,Refund request,Steps to request a refund for an eligible order.,0.104946


# Create Semantic Embeddings

In [5]:
componentCount = min(
    3,
    documentMatrix.shape[0] - 1,
    documentMatrix.shape[1] - 1,
)

semanticModel = make_pipeline(
    TruncatedSVD(
        n_components=componentCount,
        random_state=seed,
    ),
    Normalizer(copy=False),
)

documentEmbeddings = semanticModel.fit_transform(
    documentMatrix
)

print("Embedding shape:", documentEmbeddings.shape)

Embedding shape: (6, 3)


# Search Semantic Space

In [6]:
def searchSemantic(query, topK=3):
    queryVector = vectorizer.transform([query])

    queryEmbedding = semanticModel.transform(
        queryVector
    )

    scores = cosine_similarity(
        queryEmbedding,
        documentEmbeddings,
    ).ravel()

    results = faq[
        ["doc_id", "title", "text"]
    ].copy()

    results["semantic_score"] = scores

    return (
        results
        .sort_values(
            "semantic_score",
            ascending=False,
        )
        .head(topK)
        .reset_index(drop=True)
    )


semanticResults = searchSemantic(query)
display(semanticResults)

,doc_id,title,text,semantic_score
0,5,Delivery tracking,Track the current delivery status of your order.,0.989700
1,2,Failed payment,What to do when money is deducted but the paym...,0.872156
2,3,Refund request,Steps to request a refund for an eligible order.,0.489228


# Compare Search Results

In [7]:
comparison = (
    tfidfResults[
        ["doc_id", "title", "tfidf_score"]
    ]
    .merge(
        semanticResults[
            ["doc_id", "semantic_score"]
        ],
        on="doc_id",
        how="outer",
    )
    .sort_values(
        "semantic_score",
        ascending=False,
    )
)

print("Query:", query)
display(comparison.round(4))

Query: money taken but order failed


,doc_id,title,tfidf_score,semantic_score
2,5,Delivery tracking,0.1074,0.9897
0,2,Failed payment,0.3393,0.8722
1,3,Refund request,0.1049,0.4892


# Test Additional Queries

In [8]:
testQueries = [
    "I forgot my login password",
    "where is my package",
    "the item arrived broken",
]

rows = []

for testQuery in testQueries:
    topResult = searchSemantic(
        testQuery,
        topK=1,
    ).iloc[0]

    rows.append({
        "query": testQuery,
        "best_match": topResult["title"],
        "score": topResult["semantic_score"],
    })

display(
    pd.DataFrame(rows).round(4)
)

,query,best_match,score
0,I forgot my login password,Account verification,0.000
1,where is my package,Reset password,0.000
2,the item arrived broken,Damaged product,0.999
